# Modular GNN+PPO scheduler — Approach A (team as a node type)

This is a refactor of `4-teams_1.ipynb` that makes the network **invariant to the number of teams and employees**, so a checkpoint trained on one scenario (e.g. 4 teams / 24 emp) can be loaded onto another (e.g. 2 teams / 12 emp).

**What changed vs. the original**
- `team` is now a first-class node type in the heterograph, with `member_of` (employee↔team) and `demands` (team↔day) relations.
- Team identity is encoded **relationally** (via the team node embedding), not positionally — so no weight tensor's shape depends on `num_teams`.
- Feature dims are slimmed: employee `[days_worked, streak, position]` (dim 3), day `[is_special, position]` (dim 2), team `[size, demand_today]` (dim 2). The per-`(shift,team)` coverage matrix and per-team one-hots are gone.
- The action heads condition on `team_emb[team_id]` + a single live `gap` scalar instead of the flattened coverage vector + team one-hot. `head_in = 3·encoded_dim + num_shifts + 2` — independent of team/employee count.
- Per-day GNN caching, constraint masking and the pointer mechanism are all preserved.
- Adds `load_transferable_weights()` for cross-scenario warm-starts.

In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
pip install "numpy<2.0.0"

In [ ]:
pip install holidays

In [14]:
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path

DEMAND = 0
CAPACITY = 1

def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnv(gym.Env):
    def __init__(self,
                 data_dir: str = "../../../../data/problems/SMARTASK_4TEAMS_24EMP",
                 capacity_slack: int = 2):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(teams) > 1 for teams in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])

        self.special_days = _build_special_days(self.year, self.num_days)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22
        self.capacity_slack = capacity_slack

        self.PASS_ACTION = self.num_employees
        self.action_space = gym.spaces.Discrete(self.num_employees + 1)
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        self.reset()

    def build_slot_queue(self, min_demand, capacity_slack=2):
        num_days, num_shifts, num_teams = min_demand.shape
        demand_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    headcount = int(min_demand[d, s, t])
                    for _ in range(headcount):
                        demand_slots.append((d, s, t, DEMAND))

        capacity_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    for _ in range(capacity_slack):
                        capacity_slots.append((d, s, t, CAPACITY))

        return demand_slots + capacity_slots, len(demand_slots)

    def _get_assigned_shift(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return None
        return self.shift_codes[s]

    def _get_prev_shift(self, emp, day):
        if day == 0:
            return None
        return self._get_assigned_shift(emp, day - 1)

    def _get_next_shift(self, emp, day):
        if day >= self.num_days - 1:
            return None
        return self._get_assigned_shift(emp, day + 1)

    def _consecutive_streak_if_work(self, emp, day):
        streak = 1
        d = day - 1
        while d >= 0 and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d += 1
        return streak

    def _get_info(self):
        return {}

    def _obs(self):
        return np.zeros(1, dtype=np.float32)

    def current_slot(self):
        if self.slot_idx >= len(self.slot_queue):
            return None
        return self.slot_queue[self.slot_idx]

    def current_day(self):
        s = self.current_slot()
        return s[0] if s is not None else 0

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _calculate_reward(self, action, day, s_idx, t_idx, kind):
        if action == self.PASS_ACTION:
            return -5.0 if kind == DEMAND else 0.0
        team = self.teams[t_idx]
        bonus = self._team_balance_bonus(action, team)
        if kind == DEMAND:
            return 1.0 + 0.2 * bonus
        if self.days_worked[action] >= 200:
            return 0.25 + 0.1 * bonus
        return 0.5 + 0.1 * bonus

    def _calculate_final_reward(self):
        shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
        return 200.0 if shortfall == 0 else 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        self.daily_coverage = np.zeros(
            (self.num_days, self.num_shifts, self.num_teams), dtype=np.float32
        )
        self.emp_day_shift = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.emp_day_team = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.slot_queue, self.num_demand_slots = self.build_slot_queue(self.min_demand, self.capacity_slack)
        self.slot_idx = 0
        self.demand_skips = 0
        return self._obs(), self._get_info()

    def step(self, action):
        if self.slot_idx >= len(self.slot_queue):
            return self._obs(), 0.0, True, False, self._get_info()
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        action = int(action)

        if action != self.PASS_ACTION:
            self.emp_day_shift[action, day] = s_idx
            self.emp_day_team[action, day] = t_idx
            self.days_worked[action] += 1
            self.daily_coverage[day, s_idx, t_idx] += 1
            if day in self.special_days:
                self.special_days_worked[action] += 1
        elif kind == DEMAND:
            self.demand_skips += 1

        reward = self._calculate_reward(action, day, s_idx, t_idx, kind)
        self.slot_idx += 1
        terminated = self.slot_idx >= len(self.slot_queue)
        if terminated:
            reward += self._calculate_final_reward()
        return self._obs(), reward, terminated, False, self._get_info()

    def _emp_can_cover(self, emp_id, day_id, shift_id, team_id):
        if self.teams[team_id] not in self.employee_teams[emp_id]:
            return False
        if self.vac_mask[emp_id, day_id]:
            return False
        if self.emp_day_shift[emp_id, day_id] >= 0:
            return False
        if self.days_worked[emp_id] >= self.max_days_per_year:
            return False
        if day_id in self.special_days and self.special_days_worked[emp_id] >= self.special_days_cap:
            return False
        if self._consecutive_streak_if_work(emp_id, day_id) > self.max_consecutive_days:
            return False

        prev_shift = self._get_prev_shift(emp_id, day_id)
        next_shift = self._get_next_shift(emp_id, day_id)
        shift_code = self.shift_codes[shift_id]

        if shift_code == "M" and prev_shift == "T":
            return False
        if shift_code == "T" and next_shift == "M":
            return False

        return True

    def get_employee_mask(self):
        mask = np.zeros(self.num_employees + 1, dtype=bool)
        if self.slot_idx >= len(self.slot_queue):
            mask[self.PASS_ACTION] = True
            return mask
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        for e in range(self.num_employees):
            if self._emp_can_cover(e, day, s_idx, t_idx):
                mask[e] = True
        if kind == CAPACITY:
            mask[self.PASS_ACTION] = True
        elif not mask[:self.num_employees].any():
            mask[self.PASS_ACTION] = True
        return mask

    def action_label(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return "-"
        t = self.emp_day_team[emp, day]
        return f"{self.shift_codes[s]}-{self.teams[t]}"

    def render(self):
        for emp in range(self.num_employees):
            schedule = [self.action_label(emp, day) for day in range(self.num_days)]
            print(f"Employee {emp + 1:2d}: {schedule}")

In [15]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np


def emp_feat_dim(env):
    # [days_worked/223, streak/5, emp_position]  -- team membership moved to edges
    return 3


def day_feat_dim(env):
    # [is_special, day_position]  -- coverage moved to team nodes / per-slot gap
    return 2


def team_feat_dim(env):
    # [team_size/num_employees, team_demand_today/10]
    return 2


def build_graph(env):
    # employee <-> day: complete bipartite candidate edges (as in the original)
    ed_src, ed_dst = [], []
    for emp in range(env.num_employees):
        for day in range(env.num_days):
            ed_src.append(emp)
            ed_dst.append(day)

    # employee <-> team: membership (replaces the per-team one-hot feature)
    et_src, et_dst = [], []
    for emp in range(env.num_employees):
        for t_idx, team in enumerate(env.teams):
            if team in env.employee_teams[emp]:
                et_src.append(emp)
                et_dst.append(t_idx)

    # team <-> day: demand (team has >=1 required slot that day)
    td_src, td_dst = [], []
    for t_idx in range(env.num_teams):
        for day in range(env.num_days):
            if env.min_demand[day, :, t_idx].sum() > 0:
                td_src.append(t_idx)
                td_dst.append(day)

    graph_data = {
        ("employee", "assigned_to", "day"): (torch.tensor(ed_src), torch.tensor(ed_dst)),
        ("day", "staffed_by", "employee"):  (torch.tensor(ed_dst), torch.tensor(ed_src)),
        ("employee", "member_of", "team"):  (torch.tensor(et_src), torch.tensor(et_dst)),
        ("team", "has_member", "employee"): (torch.tensor(et_dst), torch.tensor(et_src)),
        ("team", "demands", "day"):         (torch.tensor(td_src), torch.tensor(td_dst)),
        ("day", "demanded_by", "team"):     (torch.tensor(td_dst), torch.tensor(td_src)),
    }
    g = dgl.heterograph(graph_data, num_nodes_dict={
        "employee": env.num_employees,
        "day": env.num_days,
        "team": env.num_teams,
    })
    update_graph_features(g, env)
    return g


def update_graph_features(g, env):
    day = env.current_day()

    emp_feats = np.zeros((env.num_employees, emp_feat_dim(env)), dtype=np.float32)
    for emp in range(env.num_employees):
        emp_feats[emp, 0] = env.days_worked[emp] / 223.0
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, day) / 5.0
        emp_feats[emp, 2] = emp / env.num_employees

    day_feats = np.zeros((env.num_days, day_feat_dim(env)), dtype=np.float32)
    for d in range(env.num_days):
        day_feats[d, 0] = float(d in env.special_days)
        day_feats[d, 1] = d / env.num_days

    team_feats = np.zeros((env.num_teams, team_feat_dim(env)), dtype=np.float32)
    for t_idx, team in enumerate(env.teams):
        team_feats[t_idx, 0] = env.team_sizes[team] / env.num_employees
        gap_today = float(np.maximum(
            0, env.min_demand[day, :, t_idx] - env.daily_coverage[day, :, t_idx]).sum())
        team_feats[t_idx, 1] = gap_today / 10.0

    g.nodes["employee"].data["feat"] = torch.tensor(emp_feats, dtype=torch.float32)
    g.nodes["day"].data["feat"] = torch.tensor(day_feats, dtype=torch.float32)
    g.nodes["team"].data["feat"] = torch.tensor(team_feats, dtype=torch.float32)


def slot_gap(env, day_id, s_idx, t_idx):
    # live coverage gap for THIS (day, shift, team) slot -- a single scalar.
    return float(env.min_demand[day_id, s_idx, t_idx] - env.daily_coverage[day_id, s_idx, t_idx])

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GNNActorCritic(nn.Module):
    def __init__(self, emp_in_feats, day_in_feats, team_in_feats, num_shifts,
                 hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.num_shifts = num_shifts

        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.day_proj = nn.Linear(day_in_feats, hidden_dim)
        self.team_proj = nn.Linear(team_in_feats, hidden_dim)

        self.conv1 = dglnn.HeteroGraphConv({
            "assigned_to": dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
            "staffed_by":  dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
            "member_of":   dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
            "has_member":  dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
            "demands":     dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
            "demanded_by": dglnn.SAGEConv(hidden_dim, hidden_dim, "mean"),
        }, aggregate="sum")

        self.conv2 = dglnn.HeteroGraphConv({
            "assigned_to": dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
            "staffed_by":  dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
            "member_of":   dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
            "has_member":  dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
            "demands":     dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
            "demanded_by": dglnn.SAGEConv(hidden_dim, encoded_dim, "mean"),
        }, aggregate="sum")

        # ctx = [day_emb, team_emb, gap_scalar, shift_oh, kind]
        # -> all dims fixed (independent of num_teams / num_employees)
        ctx_dim = 2 * encoded_dim + 1 + num_shifts + 1
        head_in = encoded_dim + ctx_dim

        self.score_head = nn.Sequential(
            nn.Linear(head_in, 64), nn.Tanh(), nn.Linear(64, 1))
        self.pass_head = nn.Sequential(
            nn.Linear(head_in, 64), nn.Tanh(), nn.Linear(64, 1))
        self.critic_head = nn.Sequential(
            nn.Linear(head_in, 64), nn.Tanh(), nn.Linear(64, 1))

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        return cls(
            emp_in_feats=emp_feat_dim(env),
            day_in_feats=day_feat_dim(env),
            team_in_feats=team_feat_dim(env),
            num_shifts=env.num_shifts,
            hidden_dim=hidden_dim,
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "day":      self.day_proj(g.nodes["day"].data["feat"]),
            "team":     self.team_proj(g.nodes["team"].data["feat"]),
        }
        h = self.conv1(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["day"], h["team"]

    def _heads(self, emp_emb, day_emb, team_emb,
               day_ids, team_ids, gap, shift, kind, action_masks):
        E = emp_emb.shape[0]
        ctx = torch.cat([day_emb[day_ids], team_emb[team_ids],
                         gap / 10.0, shift, kind], dim=-1)            # (B, ctx_dim)
        B = ctx.shape[0]

        emp_b = emp_emb.unsqueeze(0).expand(B, E, -1)
        ctx_b = ctx.unsqueeze(1).expand(B, E, -1)
        emp_logits = self.score_head(torch.cat([emp_b, ctx_b], dim=-1)).squeeze(-1)  # (B, E)

        pooled = torch.cat([emp_emb.mean(0, keepdim=True).expand(B, -1), ctx], dim=-1)
        pass_logit = self.pass_head(pooled)                                          # (B, 1)
        values = self.critic_head(pooled).squeeze(-1)                                # (B,)

        logits = torch.cat([emp_logits, pass_logit], dim=-1)                         # (B, E+1)
        masks_bool = torch.as_tensor(action_masks, dtype=torch.bool)
        if masks_bool.dim() == 1:
            masks_bool = masks_bool.unsqueeze(0)
        all_invalid = ~masks_bool.any(dim=-1)
        if all_invalid.any():
            masks_bool = masks_bool.clone()
            masks_bool[all_invalid, -1] = True
        logits = logits.masked_fill(~masks_bool, float("-inf"))
        return F.softmax(logits, dim=-1), values

    def forward(self, g, day_ids, team_ids, gap, shift, kind, action_masks):
        emp_emb, day_emb, team_emb = self.gnn_forward(g)
        return self._heads(emp_emb, day_emb, team_emb,
                           day_ids, team_ids, gap, shift, kind, action_masks)

In [17]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    day_ids: list = field(default_factory=list)
    team_ids: list = field(default_factory=list)
    gap_scalars: list = field(default_factory=list)
    shifts: list = field(default_factory=list)
    kinds: list = field(default_factory=list)
    action_masks: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    snap_ids: list = field(default_factory=list)
    emp_feats_unique: list = field(default_factory=list)
    day_feats_unique: list = field(default_factory=list)
    team_feats_unique: list = field(default_factory=list)

    def to_tensors(self):
        return {
            "day_ids": torch.tensor(self.day_ids, dtype=torch.long),
            "team_ids": torch.tensor(self.team_ids, dtype=torch.long),
            "gaps": torch.tensor(self.gap_scalars, dtype=torch.float32).unsqueeze(-1),
            "shifts": torch.stack(self.shifts),
            "kinds": torch.stack(self.kinds),
            "action_masks": torch.stack(self.action_masks),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "snap_ids": torch.tensor(self.snap_ids, dtype=torch.long),
            "emp_feats_unique": torch.stack(self.emp_feats_unique),
            "day_feats_unique": torch.stack(self.day_feats_unique),
            "team_feats_unique": torch.stack(self.team_feats_unique),
        }


def collect_trajectory(env, model, graph):
    model.eval()
    traj = Trajectory()
    env.reset()
    terminated = truncated = False

    cached_day_id = None
    cached_emp_emb = cached_day_emb = cached_team_emb = None
    cur_snap = -1

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            # GNN embeddings depend on per-day state -> recompute only when the day changes.
            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id
                traj.emp_feats_unique.append(graph.nodes["employee"].data["feat"].clone())
                traj.day_feats_unique.append(graph.nodes["day"].data["feat"].clone())
                traj.team_feats_unique.append(graph.nodes["team"].data["feat"].clone())
                cur_snap = len(traj.emp_feats_unique) - 1

            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.tensor([[gap]], dtype=torch.float32)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, values = model._heads(
                cached_emp_emb, cached_day_emb, cached_team_emb,
                day_id_t, team_id_t, gap_t, shift_t, kind_t, mask_t,
            )

            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.day_ids.append(day_id)
            traj.team_ids.append(t_idx)
            traj.gap_scalars.append(gap)
            traj.shifts.append(shift_t[0])
            traj.kinds.append(kind_t[0])
            traj.action_masks.append(mask_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.snap_ids.append(cur_snap)

    return traj

In [18]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values,  dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )

In [19]:
from torch.utils.data.sampler import BatchSampler, SubsetRandomSampler


def ppo_update(model, optimizer, batch, advantages, returns, graph,
               clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
               K_epochs=4, mini_batch_size=512):
    T = batch["actions"].shape[0]
    losses, track_entropy = [], []
    model.train()

    for _ in range(K_epochs):
        for index in BatchSampler(SubsetRandomSampler(range(T)), mini_batch_size, False):
            idx = torch.tensor(index)

            snap0 = batch["snap_ids"][idx[0]]
            graph.nodes["employee"].data["feat"] = batch["emp_feats_unique"][snap0]
            graph.nodes["day"].data["feat"]      = batch["day_feats_unique"][snap0]
            graph.nodes["team"].data["feat"]     = batch["team_feats_unique"][snap0]
            emp_emb, day_emb, team_emb = model.gnn_forward(graph)

            probs, values = model._heads(
                emp_emb, day_emb, team_emb,
                batch["day_ids"][idx], batch["team_ids"][idx], batch["gaps"][idx],
                batch["shifts"][idx], batch["kinds"][idx], batch["action_masks"][idx],
            )
            dist = Categorical(probs=probs)
            logp = dist.log_prob(batch["actions"][idx])
            ratios = torch.exp(logp - batch["log_probs_old"][idx])
            adv = advantages[idx]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist.entropy().mean()
            critic_loss = F.smooth_l1_loss(values, returns[idx])
            loss = actor_loss + value_coeff * critic_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())
            track_entropy.append(dist.entropy().mean().item())

    return float(np.mean(losses)), float(np.mean(track_entropy))

In [20]:
import os

# Hyperparameters
NUM_EPISODES = 10000
GAMMA = 0.99
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.05
ENTROPY_MIN = 0.005
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 3e-4
SAVE_EVERY = 50

BEST_CKPT = "best_4teams_modular.pth"
LATEST_CKPT = "latest_4teams_modular.pth"
RESUME_FROM = None  # set to LATEST_CKPT to resume

env = ScheduleEnv()
graph = build_graph(env)
# Model dims depend only on hidden/encoded sizes and num_shifts -> any team/employee count loads.
model = GNNActorCritic.from_env(env)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

print(f"Scenario: {env.num_teams} teams ({env.teams}), {env.num_shifts} shifts ({env.shift_codes}), "
      f"{env.num_employees} employees, num_actions={env.num_employees + 1}")

best_reward = -float("inf")
start_episode = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_episode = ckpt["episode"]
    best_reward = float(ckpt.get("best_reward", -float("inf")))
    print(f"Resumed from episode {start_episode}, best reward = {best_reward:.1f}")

for episode in range(start_episode, NUM_EPISODES):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (0.999 ** episode))

    traj = collect_trajectory(env, model, graph)
    batch = traj.to_tensors()

    advantages, returns = compute_gae(traj.rewards, traj.values, GAMMA, LAM)

    mean_loss, mean_entropy = ppo_update(
        model, optimizer, batch, advantages, returns, graph,
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
    )

    total_reward = sum(traj.rewards)
    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    demand_skips = env.demand_skips
    days_worked_str = ",".join(str(int(d)) for d in env.days_worked)
    mean_days = env.days_worked.mean()

    if total_reward > best_reward:
        best_reward = float(total_reward)
        torch.save(
            {"episode": episode, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            BEST_CKPT,
        )
        print(f"  NEW BEST: {best_reward:.1f} at ep {episode} "
              f"| shortfall={shortfall} | demand_skips={demand_skips} | days={mean_days:.0f}")

    if episode % 10 == 0:
        print(f"Ep {episode:>4} | R={total_reward:>8.1f} | Best={best_reward:>8.1f} "
              f"| Loss={mean_loss:.4f} | ent={mean_entropy:.4f} "
              f"| shortfall={shortfall} | skips={demand_skips} "
              f"| days={mean_days:.0f} [{days_worked_str}]")

    if (episode + 1) % SAVE_EVERY == 0:
        torch.save(
            {"episode": episode + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_reward": best_reward},
            LATEST_CKPT,
        )

Scenario: 4 teams (['A', 'B', 'C', 'D']), 2 shifts (['M', 'T']), 24 employees, num_actions=25


KeyboardInterrupt: 

In [21]:
def evaluate(model, env, graph, greedy=False):
    model.eval()
    env.reset()
    terminated = truncated = False
    total_reward = 0.0

    cached_day_id = None
    cached_emp_emb = cached_day_emb = cached_team_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id

            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.tensor([[gap]], dtype=torch.float32)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, _ = model._heads(
                cached_emp_emb, cached_day_emb, cached_team_emb,
                day_id_t, team_id_t, gap_t, shift_t, kind_t, mask_t,
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    snapshot = {"demand_skips": env.demand_skips, "daily_coverage": env.daily_coverage.copy()}
    schedule = [
        [env.action_label(emp, day) for day in range(env.num_days)]
        for emp in range(env.num_employees)
    ]
    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot


ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint: episode {ckpt['episode']}, best reward = {ckpt['best_reward']:.1f}\n")

N = 20
results = []
for i in range(N):
    torch.manual_seed(42 + i)
    reward, shortfall, days_worked, schedule, snap = evaluate(model, env, graph, greedy=False)
    results.append((reward, shortfall, days_worked, schedule, snap))
    print(f"Run {i+1:2d} | Reward: {reward:8.1f} | Shortfall: {shortfall:3d} "
          f"| demand_skips: {snap['demand_skips']:3d} "
          f"| Days worked: {days_worked} (avg={np.mean(days_worked):.0f})")

best_idx = min(range(N), key=lambda i: results[i][1])
best_reward, best_shortfall, best_days, best_schedule, best_snap = results[best_idx]

print(f"\n{'='*60}")
print(f"BEST: Run {best_idx+1} | Reward: {best_reward:.1f} | Shortfall: {best_shortfall}")
print(f"Days worked/employee: {best_days} (avg={np.mean(best_days):.0f})")
print(f"{'='*60}\n")

import csv
schedule_csv = f"best_schedule_{env.num_teams}teams_modular.csv"
with open(schedule_csv, "w", newline="") as f:
    writer = csv.writer(f)
    header = ["Employee"] + [f"Day_{d}" for d in range(len(best_schedule[0]))]
    writer.writerow(header)
    for emp in range(len(best_schedule)):
        writer.writerow([f"Employee_{emp+1}"] + best_schedule[emp])

print(f"Best schedule written to {schedule_csv}")

Checkpoint: episode 30, best reward = 4192.0



KeyboardInterrupt: 

## Cross-scenario transfer

Because no weight tensor's shape depends on `num_teams` or `num_employees`, a checkpoint trained on one scenario loads onto another. `load_transferable_weights` copies every shape-matching tensor and reports any it had to skip (there should be none when only team/employee counts differ; skips would only appear if `num_shifts` changed).

In [ ]:
def load_transferable_weights(model, ckpt_path, verbose=True):
    """Copy every shape-matching tensor from a checkpoint into `model`.

    With Approach A all tensors match across team/employee counts, so this is a
    full transfer. Mismatches (e.g. a different num_shifts) are skipped and left
    at their freshly-initialized values.
    """
    ckpt = torch.load(ckpt_path, weights_only=True)
    src = ckpt["model_state_dict"]
    tgt = model.state_dict()
    loaded, skipped = [], []
    for k, v in src.items():
        if k in tgt and tgt[k].shape == v.shape:
            tgt[k] = v
            loaded.append(k)
        else:
            skipped.append(k)
    model.load_state_dict(tgt)
    if verbose:
        print(f"Transferred {len(loaded)}/{len(src)} tensors.")
        if skipped:
            print(f"Skipped (shape mismatch / missing): {skipped}")
    return model


# Example: warm-start a 2-team / 12-emp model from the 4-team checkpoint, then evaluate.
env2 = ScheduleEnv(data_dir="../../../../data/problems/SMARTASK_SIMPLE_2025")
graph2 = build_graph(env2)
model2 = GNNActorCritic.from_env(env2)
load_transferable_weights(model2, BEST_CKPT)
best = min(
      (evaluate(model2, env2, graph2, greedy=False) for _ in range(20)),
      key=lambda r: r[1],   # r[1] = shortfall
)
print("best-of-20 shortfall:", best[1], "reward:", best[0])

Transferred 54/54 tensors.
best-of-20 shortfall: 38 reward: 2285.25


In [ ]:
print("total demand:", int(env2.min_demand.sum()), "| floor 16 | zero-shot 38")


total demand: 2086 | floor 16 | zero-shot 38


## Multi-task (generalist) training

Train one shared model across a **pool** of problems (domain randomization) so it generalizes to unseen scenarios with a smaller zero-shot gap. Approach A is what makes this possible: identical weight shapes across team/employee counts, so a single set of weights ingests 2-, 4-, 8-, 16-team problems interchangeably.

**Design:**
- A problem is sampled each episode (**interleaved** → avoids catastrophic forgetting).
- The **graded terminal reward** (next cell) puts every problem on the same 0–200 scale; without it the critic chases very different reward magnitudes across instances.
- `16TEAMS_96EMP` is **held out** and evaluated zero-shot during training to track generalization.
- Pipeline: **generalist pretrain → optional short per-instance fine-tune**. The generalist won't beat a per-instance specialist on any single problem, but it should be a near-optimal *starting point* on unseen ones.

**Run order:** run the env / graph / model / training-function cells above first (so `ScheduleEnv`, `build_graph`, `collect_trajectory`, `compute_gae`, `ppo_update`, `evaluate`, `GNNActorCritic`, `load_transferable_weights`, and `BEST_CKPT` exist), then the four cells below in order.

> **Cost note:** 8- and 16-team episodes have far more slots than 2-team ones, so wall-clock per episode varies a lot. If it's too slow, shrink the pool or lower `MT_EPISODES` / `EVAL_EVERY`.

In [22]:
# Graded terminal reward: rewards partial coverage so progress is signalled even on
# instances whose optimal shortfall > 0, and puts different-sized problems on one 0-200
# scale (so the critic isn't chasing wildly different reward magnitudes across the pool).
# Monkeypatch so we don't disturb the original ScheduleEnv cell; fold into the class for
# a final version. Affects all instances, since the method is resolved at call time.
def _graded_final_reward(self):
    shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
    total = int(self.min_demand.sum())
    return 200.0 * (1.0 - shortfall / max(total, 1))

ScheduleEnv._calculate_final_reward = _graded_final_reward
print("Patched ScheduleEnv._calculate_final_reward -> graded coverage reward")


Patched ScheduleEnv._calculate_final_reward -> graded coverage reward


In [24]:
import random

# --- problem pool (hold 16 teams out for generalization testing) ---
TRAIN_DIRS = [
    "../../../../data/problems/SMARTASK_2TEAMS_12EMP",
    "../../../../data/problems/SMARTASK_4TEAMS_24EMP",
    "../../../../data/problems/SMARTASK_8TEAMS_48EMP",
]
HELDOUT_DIR = "../../../../data/problems/SMARTASK_16TEAMS_96EMP"

MT_EPISODES   = 6000
MT_LR         = 3e-4
GAMMA, LAM    = 0.99, 0.95
CLIP_EPS      = 0.15
VALUE_COEFF   = 0.5
ENT0, ENT_MIN = 0.05, 0.005
K_EPOCHS      = 3
MINI_BATCH    = 512
EVAL_EVERY    = 50
GEN_BEST_CKPT = "best_generalist.pth"

# Build envs + static graphs once. Bigger scenarios (8/16 teams) make episodes heavier;
# drop one from the pool or weight the sampler if wall-clock gets painful.
train_envs    = [ScheduleEnv(data_dir=d) for d in TRAIN_DIRS]
train_graphs  = [build_graph(e) for e in train_envs]
heldout_env   = ScheduleEnv(data_dir=HELDOUT_DIR)
heldout_graph = build_graph(heldout_env)

# Dims are identical across scenarios, so build the shared model from any env.
gen_model = GNNActorCritic.from_env(train_envs[0])
gen_opt   = torch.optim.Adam(gen_model.parameters(), lr=MT_LR, eps=1e-5)

def _coverage(env):
    sf = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    return 1.0 - sf / max(int(env.min_demand.sum()), 1), sf

def _heldout_zero_shot(n=5):
    sf = min(evaluate(gen_model, heldout_env, heldout_graph, greedy=False)[1] for _ in range(n))
    return 1.0 - sf / max(int(heldout_env.min_demand.sum()), 1), sf

print(f"Pool: {[d.split('/')[-1] for d in TRAIN_DIRS]} | held-out: {HELDOUT_DIR.split('/')[-1]}")

best_heldout = -1.0
for ep in range(MT_EPISODES):
    i = random.randrange(len(train_envs))            # sample a problem (interleaved)
    env, graph = train_envs[i], train_graphs[i]
    ent = max(ENT_MIN, ENT0 * (0.999 ** ep))

    traj = collect_trajectory(env, gen_model, graph)
    batch = traj.to_tensors()
    adv, ret = compute_gae(traj.rewards, traj.values, GAMMA, LAM)
    loss, entropy = ppo_update(gen_model, gen_opt, batch, adv, ret, graph,
                               clip_eps=CLIP_EPS, value_coeff=VALUE_COEFF,
                               entropy_coeff=ent, K_epochs=K_EPOCHS, mini_batch_size=MINI_BATCH)

    if ep % 20 == 0:
        cov, sf = _coverage(env)
        print(f"ep {ep:5d} | {TRAIN_DIRS[i].split('/')[-1]:22s} | cov={cov:.4f} sf={sf:4d} "
              f"| loss={loss:.3f} ent={entropy:.3f}")

    if ep > 0 and ep % EVAL_EVERY == 0:
        hcov, hsf = _heldout_zero_shot()
        flag = ""
        if hcov > best_heldout:
            best_heldout = hcov
            torch.save({"episode": ep, "model_state_dict": gen_model.state_dict(),
                        "heldout_coverage": hcov}, GEN_BEST_CKPT)
            flag = "  <- new best, saved"
        print(f"  >>> held-out zero-shot cov={hcov:.4f} sf={hsf}{flag}")


Pool: ['SMARTASK_2TEAMS_12EMP', 'SMARTASK_4TEAMS_24EMP', 'SMARTASK_8TEAMS_48EMP'] | held-out: SMARTASK_16TEAMS_96EMP
ep     0 | SMARTASK_2TEAMS_12EMP  | cov=0.9933 sf=  14 | loss=3.246 ent=0.657
ep    20 | SMARTASK_8TEAMS_48EMP  | cov=0.9993 sf=   4 | loss=2.386 ent=0.880
ep    40 | SMARTASK_4TEAMS_24EMP  | cov=0.9949 sf=  15 | loss=2.150 ent=0.876
  >>> held-out zero-shot cov=0.9990 sf=12  <- new best, saved
ep    60 | SMARTASK_4TEAMS_24EMP  | cov=0.9979 sf=   6 | loss=1.903 ent=0.957
ep    80 | SMARTASK_4TEAMS_24EMP  | cov=0.9969 sf=   9 | loss=1.586 ent=0.889
ep   100 | SMARTASK_4TEAMS_24EMP  | cov=0.9986 sf=   4 | loss=1.284 ent=0.887
  >>> held-out zero-shot cov=0.9990 sf=12
ep   120 | SMARTASK_4TEAMS_24EMP  | cov=0.9993 sf=   2 | loss=0.857 ent=0.871
ep   140 | SMARTASK_4TEAMS_24EMP  | cov=0.9983 sf=   5 | loss=0.522 ent=0.861
  >>> held-out zero-shot cov=0.9993 sf=8  <- new best, saved
ep   160 | SMARTASK_2TEAMS_12EMP  | cov=0.9938 sf=  13 | loss=0.789 ent=0.662
ep   180 | SMART

KeyboardInterrupt: 

In [ ]:
# --- Optional per-instance fine-tune + final generalization comparison ---

def finetune(env, graph, model, episodes=200, lr=1e-4, ckpt_path="finetuned.pth",
             clip_eps=0.15, value_coeff=0.5, ent0=0.02, ent_min=0.005,
             k_epochs=3, mini_batch=512, log_every=10):
    """Specialize a (warm-started) model on one problem. Lower LR / entropy than
    from-scratch: we are refining good features, not learning from noise."""
    opt = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-5)
    best_r, best_sf = -float("inf"), None
    for ep in range(episodes):
        ent = max(ent_min, ent0 * (0.999 ** ep))
        traj = collect_trajectory(env, model, graph)
        batch = traj.to_tensors()
        adv, ret = compute_gae(traj.rewards, traj.values, GAMMA, LAM)
        loss, entropy = ppo_update(model, opt, batch, adv, ret, graph,
                                   clip_eps=clip_eps, value_coeff=value_coeff,
                                   entropy_coeff=ent, K_epochs=k_epochs, mini_batch_size=mini_batch)
        sf = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
        r = sum(traj.rewards)
        if r > best_r:
            best_r, best_sf = r, sf
            torch.save({"episode": ep, "model_state_dict": model.state_dict(),
                        "best_reward": best_r}, ckpt_path)
        if ep % log_every == 0:
            print(f"ft ep {ep:4d} | R={r:8.1f} | sf={sf:4d} | best_sf={best_sf} | loss={loss:.3f}")
    return model


def best_of_n(model, env, graph, n=20):
    sf = min(evaluate(model, env, graph, greedy=False)[1] for _ in range(n))
    total = int(env.min_demand.sum())
    return 1.0 - sf / max(total, 1), sf, total


# Generalist (trained across the pool) vs single-problem checkpoint, both zero-shot
# on the held-out problem.
ckpt = torch.load(GEN_BEST_CKPT, weights_only=True)
gen_model.load_state_dict(ckpt["model_state_dict"])
gcov, gsf, total = best_of_n(gen_model, heldout_env, heldout_graph, n=20)
print(f"GENERALIST  zero-shot on {HELDOUT_DIR.split('/')[-1]}: "
      f"shortfall={gsf}/{total} -> coverage={gcov:.4f}")

single = GNNActorCritic.from_env(heldout_env)
load_transferable_weights(single, BEST_CKPT, verbose=False)   # BEST_CKPT = single 4-team run
scov, ssf, _ = best_of_n(single, heldout_env, heldout_graph, n=20)
print(f"SINGLE-PROB zero-shot on {HELDOUT_DIR.split('/')[-1]}: "
      f"shortfall={ssf}/{total} -> coverage={scov:.4f}")
print(f"\nGeneralization gain: {gcov - scov:+.4f} coverage ({ssf - gsf:+d} fewer unmet slots)")

# Close the last gap by specializing the generalist on the held-out instance:
# finetune(heldout_env, heldout_graph, gen_model, episodes=200, ckpt_path="heldout_finetuned.pth")
